# examples-seen-step-axis — faded example 3: Gradient Accumulation: examples_seen per Optimizer Step

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `examples-seen-step-axis`. The last cell reports your progress on the `Trainer: examples-seen step axis` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: examples-seen step axis` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`examples-seen-step-axis`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "examples-seen-step-axis"
DD_SUBTOPIC = "Trainer: examples-seen step axis"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

With gradient accumulation, each optimizer step processes `batch_size * accum_steps` real examples (the effective batch). The `examples_seen` counter must multiply by both factors so that runs with different micro-batch / accumulation splits but the same effective batch size produce identical x-axis values.

## Faded exercise 3

Implement `log_with_accum(losses, batch_size, accum_steps, wandb)`. For each loss (step from 1), compute `examples_seen = step * batch_size * accum_steps` and call `wandb.log({'loss': loss, 'examples_seen': examples_seen, 'step': step})`. Return the number of log calls.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def log_with_accum(losses, batch_size: int, accum_steps: int, wandb):
    n = 0
    for step, loss in enumerate(losses, start=1):
        examples_seen = step * batch_size * accum_steps
        wandb.log({'loss': loss, 'examples_seen': examples_seen, 'step': step})
        n += 1
    return n


def _test():
    class MockWandb:
        def __init__(self):
            self.calls = []
        def log(self, p):
            self.calls.append(dict(p))
    w = MockWandb()
    losses = [2.0, 1.5, 1.0]
    n = log_with_accum(losses, batch_size=8, accum_steps=4, wandb=w)
    assert n == 3
    for i, c in enumerate(w.calls):
        assert c['examples_seen'] == (i + 1) * 8 * 4
        assert c['step'] == i + 1
    # effective batch 32: same as bs=32 accum=1
    w2 = MockWandb()
    log_with_accum(losses, batch_size=32, accum_steps=1, wandb=w2)
    for c1, c2 in zip(w.calls, w2.calls):
        assert c1['examples_seen'] == c2['examples_seen']


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def log_with_accum(losses, batch_size: int, accum_steps: int, wandb):
    n = 0
    for step, loss in enumerate(losses, start=1):
        examples_seen = step * batch_size * accum_steps
        wandb.log({'loss': loss, 'examples_seen': examples_seen, 'step': step})
        n += 1
    return n
```
</details>